# N074 · 树的构建与序列化

**目标：** 在充分的编码约束下恢复树结构。

**先修：** N019, N069。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 前中序划分；索引映射；空节点标记；可逆编码；重复值歧义。

## 从问题到算法

遍历序列重建需要节点值唯一，否则仅靠前序和中序一般不唯一。迭代重建把尚未匹配中序的祖先放入栈：下一前序值要么是当前左孩子，要么是在若干祖先完成中序后接上的右孩子。序列化必须显式记录空指针，单纯输出值无法恢复形状；这里使用带#的前序文本，不使用eval。

## 最小实现

**本章接口：** `build_tree(preorder, inorder)`、`serialize(root)`、`deserialize(data)`

In [1]:
class TreeNode:
    def __init__(self,val=0,left=None,right=None):
        self.val,self.left,self.right=val,left,right

def tree_from_level(values):
    """Construct a finite binary tree from level order with None placeholders."""
    from collections import deque
    if not values or values[0] is None: return None
    root=TreeNode(values[0]); queue=deque([root]); i=1
    while queue and i<len(values):
        node=queue.popleft()
        if values[i] is not None: node.left=TreeNode(values[i]); queue.append(node.left)
        i+=1
        if i<len(values):
            if values[i] is not None: node.right=TreeNode(values[i]); queue.append(node.right)
            i+=1
    return root

def build_tree(preorder,inorder):
    if len(preorder)!=len(inorder) or len(set(preorder))!=len(preorder) or set(preorder)!=set(inorder):
        raise ValueError('matching traversals with distinct keys required')
    if not preorder: return None
    root=TreeNode(preorder[0]); stack=[root]; index=0
    for value in preorder[1:]:
        node=TreeNode(value)
        if stack[-1].val!=inorder[index]: stack[-1].left=node
        else:
            parent=None
            while stack and index<len(inorder) and stack[-1].val==inorder[index]: parent=stack.pop(); index+=1
            parent.right=node
        stack.append(node)
    while stack and index<len(inorder) and stack[-1].val==inorder[index]: stack.pop(); index+=1
    if stack or index!=len(inorder): raise ValueError('inconsistent traversals')
    return root

def serialize(root):
    tokens=[]; stack=[root]
    while stack:
        node=stack.pop()
        if node is None: tokens.append('#')
        else: tokens.append(str(node.val)); stack.append(node.right); stack.append(node.left)
    return ','.join(tokens)

def deserialize(data):
    tokens=data.split(',')
    def node_from(token): return None if token=='#' else TreeNode(int(token))
    root=node_from(tokens[0]); stack=[[root,0]] if root else []
    for token in tokens[1:]:
        if not stack: raise ValueError('extra tokens')
        child=node_from(token); parent,side=stack[-1]
        if side==0: parent.left=child; stack[-1][1]=1
        else: parent.right=child; stack.pop()
        if child is not None: stack.append([child,0])
    if stack: raise ValueError('truncated tree')
    return root

## 正确性、前提与复杂度

前序确定新节点出现次序，中序匹配确定何时结束一个祖先的左侧并转向右侧。序列化中每个非空节点消费一个槽、产生两个子槽，#消费一个槽，因此完整槽结构唯一解码。

**代价：** O(n)时间与O(n)辅助空间；文本长度还依赖整数位数。迭代实现可处理深链，不受递归限制。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

r=tree_from_level([1,-2,3,None,4]); text=serialize(r)
show_table(['token索引','前序token'],list(enumerate(text.split(','))))
show_table(['原编码','往返编码'],[(text,serialize(deserialize(text)))])

token索引,前序token
0,1
1,-2
2,#
3,4
4,#
5,#
6,3
7,#
8,#


原编码,往返编码
"1,-2,#,4,#,#,3,#,#","1,-2,#,4,#,#,3,#,#"


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
r=build_tree([3,9,20,15,7],[9,3,15,20,7])
assert serialize(r)=='3,9,#,#,20,15,#,#,7,#,#'
for values in [[],[0],[-1,2,3,None,4],[1,None,2,None,3]]:
    text=serialize(tree_from_level(values)); assert serialize(deserialize(text))==text
assert deserialize('#') is None
for bad in ['1,#','#,#','1,#,#,2','']:
    try: deserialize(bad)
    except ValueError: pass
    else: raise AssertionError('malformed encoding must fail')
try: build_tree([1,2,3],[3,1,2])
except ValueError: pass
else: raise AssertionError('inconsistent traversals must fail')
a=list(range(1500)); r=build_tree(a,a); assert serialize(deserialize(serialize(r)))==serialize(r)
print('N074: 所有本章断言通过')

N074: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 说明缺少空标记为何可能歧义。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 设计非法编码的明确错误。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 105. Construct Binary Tree from Preorder and Inorder Traversal（canonical）
- 106. Construct Binary Tree from Inorder and Postorder Traversal（canonical）
- 297. Serialize and Deserialize Binary Tree（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。